# Gaussian Mixture Models (GMM) Clustering

## Assignment (c): GMM Clustering using Scikit-learn

**Author:** Nitish  
**Date:** December 2024

---

## Table of Contents
1. [Introduction](#introduction)
2. [Theory and Mathematics](#theory)
3. [Dataset Preparation](#dataset)
4. [GMM Implementation](#gmm)
5. [Model Selection (BIC/AIC)](#selection)
6. [Comparison with K-Means](#comparison)
7. [Clustering Quality Metrics](#metrics)
8. [Advanced Applications](#advanced)
9. [Conclusion](#conclusion)

---

<a id='introduction'></a>
## 1. Introduction

Gaussian Mixture Models (GMM) is a probabilistic model that assumes data points are generated from a mixture of several Gaussian distributions with unknown parameters.

### Key Advantages over K-Means:
- **Soft clustering:** Provides probability of belonging to each cluster
- **Flexible cluster shapes:** Can model elliptical clusters
- **Covariance types:** Full, tied, diagonal, spherical

In [ ]:
# Install required packages
!pip install numpy pandas matplotlib seaborn scikit-learn plotly -q

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, load_iris, load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
from matplotlib.patches import Ellipse
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

print("All libraries imported successfully!")

<a id='theory'></a>
## 2. Theory and Mathematics

### Gaussian Mixture Model

A GMM represents the probability distribution of data as:

$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

Where:
- $K$ = number of components (clusters)
- $\pi_k$ = mixing coefficient (weight) for component $k$, where $\sum_k \pi_k = 1$
- $\mu_k$ = mean of component $k$
- $\Sigma_k$ = covariance matrix of component $k$
- $\mathcal{N}(x | \mu, \Sigma)$ = multivariate Gaussian distribution

### Expectation-Maximization (EM) Algorithm

1. **E-step:** Compute responsibilities (soft assignments)
   $$\gamma_{nk} = \frac{\pi_k \mathcal{N}(x_n | \mu_k, \Sigma_k)}{\sum_j \pi_j \mathcal{N}(x_n | \mu_j, \Sigma_j)}$$

2. **M-step:** Update parameters
   - $\mu_k^{new} = \frac{\sum_n \gamma_{nk} x_n}{\sum_n \gamma_{nk}}$
   - $\Sigma_k^{new} = \frac{\sum_n \gamma_{nk} (x_n - \mu_k)(x_n - \mu_k)^T}{\sum_n \gamma_{nk}}$
   - $\pi_k^{new} = \frac{\sum_n \gamma_{nk}}{N}$

### Covariance Types

| Type | Description | Parameters |
|------|-------------|------------|
| **Full** | Each component has its own general covariance matrix | Most flexible |
| **Tied** | All components share the same covariance matrix | Moderate |
| **Diagonal** | Each component has its own diagonal covariance | Axis-aligned |
| **Spherical** | Each component has single variance | Circular clusters |

<a id='dataset'></a>
## 3. Dataset Preparation

In [ ]:
# Generate synthetic datasets with different characteristics

# Dataset 1: Spherical clusters (good for K-Means)
X_spherical, y_spherical = make_blobs(n_samples=500, centers=4, 
                                       cluster_std=0.8, random_state=42)

# Dataset 2: Elongated/elliptical clusters (better for GMM)
def generate_elliptical_clusters(n_samples=500, n_clusters=3, random_state=42):
    np.random.seed(random_state)
    X = []
    y = []
    
    # Define different covariance matrices for elongated clusters
    covs = [
        [[3, 1], [1, 0.5]],
        [[0.5, -0.3], [-0.3, 2]],
        [[2, 0.8], [0.8, 1]]
    ]
    means = [[0, 0], [5, 5], [0, 8]]
    
    samples_per_cluster = n_samples // n_clusters
    
    for i in range(n_clusters):
        cluster_data = np.random.multivariate_normal(means[i], covs[i], samples_per_cluster)
        X.append(cluster_data)
        y.extend([i] * samples_per_cluster)
    
    return np.vstack(X), np.array(y)

X_elliptical, y_elliptical = generate_elliptical_clusters()

# Dataset 3: Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Dataset 4: Wine dataset
wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print("Datasets created:")
print(f"  - Spherical clusters: {X_spherical.shape}")
print(f"  - Elliptical clusters: {X_elliptical.shape}")
print(f"  - Iris: {X_iris.shape}")
print(f"  - Wine: {X_wine.shape}")

In [ ]:
# Visualize synthetic datasets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
scatter1 = ax1.scatter(X_spherical[:, 0], X_spherical[:, 1], c=y_spherical, 
                       cmap='viridis', alpha=0.7, s=50)
ax1.set_title('Spherical Clusters (4 clusters)', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

ax2 = axes[1]
scatter2 = ax2.scatter(X_elliptical[:, 0], X_elliptical[:, 1], c=y_elliptical, 
                       cmap='viridis', alpha=0.7, s=50)
ax2.set_title('Elliptical Clusters (3 clusters)', fontsize=14)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
plt.colorbar(scatter2, ax=ax2, label='Cluster')

plt.tight_layout()
plt.show()

In [ ]:
# Standardize data
scaler = StandardScaler()
X_spherical_scaled = scaler.fit_transform(X_spherical)
X_elliptical_scaled = scaler.fit_transform(X_elliptical)
X_iris_scaled = scaler.fit_transform(X_iris)
X_wine_scaled = scaler.fit_transform(X_wine)

print("Data standardized successfully!")

<a id='gmm'></a>
## 4. GMM Implementation

In [ ]:
# Fit GMM on elliptical data
gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm_labels = gmm.fit_predict(X_elliptical)

# Get probabilities
gmm_probs = gmm.predict_proba(X_elliptical)

print("GMM Model Parameters:")
print(f"  - Number of components: {gmm.n_components}")
print(f"  - Covariance type: {gmm.covariance_type}")
print(f"  - Converged: {gmm.converged_}")
print(f"  - Number of iterations: {gmm.n_iter_}")
print(f"  - Log-likelihood: {gmm.score(X_elliptical):.4f}")

In [ ]:
# Display GMM parameters
print("\n" + "=" * 60)
print("GMM LEARNED PARAMETERS")
print("=" * 60)

print("\nMixing Weights (π):")
for i, weight in enumerate(gmm.weights_):
    print(f"  Component {i}: {weight:.4f}")

print("\nMeans (μ):")
for i, mean in enumerate(gmm.means_):
    print(f"  Component {i}: {mean}")

print("\nCovariances (Σ):")
for i, cov in enumerate(gmm.covariances_):
    print(f"  Component {i}:")
    print(f"    {cov}")

In [ ]:
# Function to draw ellipse for GMM components
def draw_ellipse(position, covariance, ax=None, **kwargs):
    """Draw an ellipse with a given position and covariance"""
    ax = ax or plt.gca()
    
    # Convert covariance to principal axes
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)
    
    # Draw the ellipse
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle=angle, **kwargs))

def plot_gmm(gmm, X, labels, ax=None, title='GMM Clustering'):
    """Plot GMM clustering results with ellipses"""
    ax = ax or plt.gca()
    
    # Plot data points
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', 
                        alpha=0.6, s=40)
    
    # Plot ellipses for each component
    colors = plt.cm.viridis(np.linspace(0, 1, gmm.n_components))
    for i, (mean, cov) in enumerate(zip(gmm.means_, gmm.covariances_)):
        draw_ellipse(mean, cov, ax=ax, alpha=0.2, 
                    facecolor=colors[i], edgecolor='black', linewidth=2)
        ax.scatter(mean[0], mean[1], c='red', marker='X', s=200, 
                  edgecolors='black', linewidths=2, zorder=5)
    
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    
    return scatter

In [ ]:
# Visualize GMM clustering with ellipses
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# True labels
ax1 = axes[0]
scatter1 = ax1.scatter(X_elliptical[:, 0], X_elliptical[:, 1], c=y_elliptical, 
                       cmap='viridis', alpha=0.7, s=50)
ax1.set_title('True Labels', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

# GMM clustering
ax2 = axes[1]
scatter2 = plot_gmm(gmm, X_elliptical, gmm_labels, ax=ax2, 
                    title='GMM Clustering with Covariance Ellipses')
plt.colorbar(scatter2, ax=ax2, label='Cluster')

plt.tight_layout()
plt.show()

In [ ]:
# Soft clustering visualization - probability heatmap
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i in range(3):
    ax = axes[i]
    scatter = ax.scatter(X_elliptical[:, 0], X_elliptical[:, 1], 
                        c=gmm_probs[:, i], cmap='Reds', alpha=0.7, s=50)
    ax.set_title(f'Probability of Cluster {i}', fontsize=14)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    plt.colorbar(scatter, ax=ax, label='Probability')

plt.suptitle('Soft Clustering: Probability of Belonging to Each Cluster', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Uncertainty visualization
# Points with high uncertainty have similar probabilities across clusters
entropy = -np.sum(gmm_probs * np.log(gmm_probs + 1e-10), axis=1)
max_entropy = np.log(gmm.n_components)  # Maximum possible entropy
uncertainty = entropy / max_entropy  # Normalized uncertainty

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_elliptical[:, 0], X_elliptical[:, 1], 
                     c=uncertainty, cmap='coolwarm', alpha=0.7, s=50)
plt.colorbar(scatter, label='Uncertainty (Normalized Entropy)')
plt.title('Clustering Uncertainty\n(Red = High uncertainty, Blue = Low uncertainty)', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.tight_layout()
plt.show()

print(f"Average uncertainty: {uncertainty.mean():.4f}")
print(f"Points with high uncertainty (>0.5): {np.sum(uncertainty > 0.5)}")

<a id='selection'></a>
## 5. Model Selection (BIC/AIC)

In [ ]:
# Model selection using BIC and AIC
def gmm_model_selection(X, max_components=10, covariance_types=['full', 'tied', 'diag', 'spherical']):
    """Select optimal GMM model using BIC and AIC"""
    results = []
    
    for cov_type in covariance_types:
        for n_comp in range(1, max_components + 1):
            gmm = GaussianMixture(n_components=n_comp, covariance_type=cov_type,
                                 random_state=42, n_init=5)
            gmm.fit(X)
            
            results.append({
                'n_components': n_comp,
                'covariance_type': cov_type,
                'BIC': gmm.bic(X),
                'AIC': gmm.aic(X),
                'log_likelihood': gmm.score(X) * len(X)
            })
    
    return pd.DataFrame(results)

# Run model selection on elliptical data
results_df = gmm_model_selection(X_elliptical, max_components=8)
print("Model Selection Results (Top 10 by BIC):")
print(results_df.sort_values('BIC').head(10).to_string(index=False))

In [ ]:
# Visualize BIC and AIC
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cov_types = ['full', 'tied', 'diag', 'spherical']
colors = ['blue', 'green', 'red', 'purple']

# BIC plot
ax1 = axes[0]
for cov_type, color in zip(cov_types, colors):
    subset = results_df[results_df['covariance_type'] == cov_type]
    ax1.plot(subset['n_components'], subset['BIC'], 'o-', 
             label=cov_type, color=color, linewidth=2, markersize=8)
ax1.set_xlabel('Number of Components', fontsize=12)
ax1.set_ylabel('BIC', fontsize=12)
ax1.set_title('BIC Score (Lower is better)', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# AIC plot
ax2 = axes[1]
for cov_type, color in zip(cov_types, colors):
    subset = results_df[results_df['covariance_type'] == cov_type]
    ax2.plot(subset['n_components'], subset['AIC'], 'o-', 
             label=cov_type, color=color, linewidth=2, markersize=8)
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('AIC', fontsize=12)
ax2.set_title('AIC Score (Lower is better)', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Model Selection: BIC and AIC Comparison', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

# Find best model
best_bic = results_df.loc[results_df['BIC'].idxmin()]
best_aic = results_df.loc[results_df['AIC'].idxmin()]

print(f"\nBest model by BIC: {best_bic['n_components']} components, {best_bic['covariance_type']} covariance")
print(f"Best model by AIC: {best_aic['n_components']} components, {best_aic['covariance_type']} covariance")

In [ ]:
# Compare different covariance types
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for idx, cov_type in enumerate(cov_types):
    ax = axes[idx // 2, idx % 2]
    
    gmm_temp = GaussianMixture(n_components=3, covariance_type=cov_type, random_state=42)
    labels_temp = gmm_temp.fit_predict(X_elliptical)
    
    scatter = ax.scatter(X_elliptical[:, 0], X_elliptical[:, 1], c=labels_temp, 
                        cmap='viridis', alpha=0.6, s=40)
    
    # Draw ellipses based on covariance type
    colors_ellipse = plt.cm.viridis(np.linspace(0, 1, 3))
    for i in range(3):
        mean = gmm_temp.means_[i]
        if cov_type == 'full':
            cov = gmm_temp.covariances_[i]
        elif cov_type == 'tied':
            cov = gmm_temp.covariances_
        elif cov_type == 'diag':
            cov = np.diag(gmm_temp.covariances_[i])
        else:  # spherical
            cov = np.eye(2) * gmm_temp.covariances_[i]
        
        draw_ellipse(mean, cov, ax=ax, alpha=0.2, 
                    facecolor=colors_ellipse[i], edgecolor='black', linewidth=2)
        ax.scatter(mean[0], mean[1], c='red', marker='X', s=150, 
                  edgecolors='black', linewidths=2, zorder=5)
    
    sil = silhouette_score(X_elliptical, labels_temp)
    ax.set_title(f'{cov_type.capitalize()} Covariance\nSilhouette: {sil:.3f}', fontsize=12)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('Comparison of Covariance Types', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

<a id='comparison'></a>
## 6. Comparison with K-Means

In [ ]:
# Compare GMM vs K-Means on elliptical data
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_elliptical)

gmm_full = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm_labels = gmm_full.fit_predict(X_elliptical)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# True labels
ax1 = axes[0]
scatter1 = ax1.scatter(X_elliptical[:, 0], X_elliptical[:, 1], c=y_elliptical, 
                       cmap='viridis', alpha=0.7, s=50)
ax1.set_title('True Labels', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')

# K-Means
ax2 = axes[1]
scatter2 = ax2.scatter(X_elliptical[:, 0], X_elliptical[:, 1], c=kmeans_labels, 
                       cmap='viridis', alpha=0.7, s=50)
ax2.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            c='red', marker='X', s=200, edgecolors='black', linewidths=2)
sil_km = silhouette_score(X_elliptical, kmeans_labels)
ari_km = adjusted_rand_score(y_elliptical, kmeans_labels)
ax2.set_title(f'K-Means\nSilhouette: {sil_km:.3f}, ARI: {ari_km:.3f}', fontsize=14)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')

# GMM
ax3 = axes[2]
scatter3 = plot_gmm(gmm_full, X_elliptical, gmm_labels, ax=ax3)
sil_gmm = silhouette_score(X_elliptical, gmm_labels)
ari_gmm = adjusted_rand_score(y_elliptical, gmm_labels)
ax3.set_title(f'GMM (Full Covariance)\nSilhouette: {sil_gmm:.3f}, ARI: {ari_gmm:.3f}', fontsize=14)

plt.suptitle('K-Means vs GMM on Elliptical Clusters', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\nGMM performs better on elliptical clusters because it can model")
print("non-spherical cluster shapes through its covariance matrices.")

In [ ]:
# Quantitative comparison
print("=" * 70)
print("GMM vs K-MEANS COMPARISON")
print("=" * 70)

comparison_data = []

datasets = [
    ('Spherical', X_spherical, y_spherical, 4),
    ('Elliptical', X_elliptical, y_elliptical, 3),
    ('Iris', X_iris_scaled, y_iris, 3),
    ('Wine', X_wine_scaled, y_wine, 3)
]

for name, X, y_true, n_clusters in datasets:
    # K-Means
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    km_labels = km.fit_predict(X)
    
    # GMM
    gm = GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=42)
    gm_labels = gm.fit_predict(X)
    
    comparison_data.append({
        'Dataset': name,
        'Method': 'K-Means',
        'Silhouette': silhouette_score(X, km_labels),
        'ARI': adjusted_rand_score(y_true, km_labels),
        'NMI': normalized_mutual_info_score(y_true, km_labels)
    })
    
    comparison_data.append({
        'Dataset': name,
        'Method': 'GMM',
        'Silhouette': silhouette_score(X, gm_labels),
        'ARI': adjusted_rand_score(y_true, gm_labels),
        'NMI': normalized_mutual_info_score(y_true, gm_labels)
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

<a id='metrics'></a>
## 7. Clustering Quality Metrics

In [ ]:
def comprehensive_gmm_evaluation(X, gmm_model, labels_pred, labels_true=None, name="Dataset"):
    """Comprehensive GMM evaluation"""
    print(f"\n{'='*60}")
    print(f"GMM EVALUATION: {name}")
    print(f"{'='*60}")
    
    # Model metrics
    print("\n--- Model Metrics ---")
    print(f"Log-likelihood:          {gmm_model.score(X) * len(X):.4f}")
    print(f"BIC:                     {gmm_model.bic(X):.4f}")
    print(f"AIC:                     {gmm_model.aic(X):.4f}")
    
    # Internal metrics
    print("\n--- Internal Clustering Metrics ---")
    print(f"Silhouette Score:        {silhouette_score(X, labels_pred):.4f}")
    print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X, labels_pred):.4f}")
    print(f"Davies-Bouldin Index:    {davies_bouldin_score(X, labels_pred):.4f}")
    
    # External metrics
    if labels_true is not None:
        print("\n--- External Metrics ---")
        print(f"Adjusted Rand Index:     {adjusted_rand_score(labels_true, labels_pred):.4f}")
        print(f"Normalized Mutual Info:  {normalized_mutual_info_score(labels_true, labels_pred):.4f}")
    
    # Cluster statistics
    print("\n--- Cluster Statistics ---")
    unique, counts = np.unique(labels_pred, return_counts=True)
    for cluster, count in zip(unique, counts):
        weight = gmm_model.weights_[cluster]
        print(f"Cluster {cluster}: {count} samples ({100*count/len(labels_pred):.1f}%), Weight: {weight:.3f}")

# Evaluate on Iris
gmm_iris = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
labels_iris_gmm = gmm_iris.fit_predict(X_iris_scaled)
comprehensive_gmm_evaluation(X_iris_scaled, gmm_iris, labels_iris_gmm, y_iris, "Iris Dataset")

In [ ]:
# Evaluate on Wine
gmm_wine = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
labels_wine_gmm = gmm_wine.fit_predict(X_wine_scaled)
comprehensive_gmm_evaluation(X_wine_scaled, gmm_wine, labels_wine_gmm, y_wine, "Wine Dataset")

<a id='advanced'></a>
## 8. Advanced Applications

In [ ]:
# Density estimation and anomaly detection
# Generate data with outliers
np.random.seed(42)
X_normal = np.random.multivariate_normal([0, 0], [[1, 0.5], [0.5, 1]], 300)
X_outliers = np.random.uniform(-5, 5, (20, 2))
X_with_outliers = np.vstack([X_normal, X_outliers])

# Fit GMM
gmm_density = GaussianMixture(n_components=1, covariance_type='full', random_state=42)
gmm_density.fit(X_normal)

# Calculate log-likelihood scores
scores = gmm_density.score_samples(X_with_outliers)

# Identify anomalies (low likelihood)
threshold = np.percentile(scores, 5)  # Bottom 5%
anomalies = scores < threshold

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scatter = plt.scatter(X_with_outliers[:, 0], X_with_outliers[:, 1], 
                     c=scores, cmap='viridis', alpha=0.7, s=50)
plt.colorbar(scatter, label='Log-likelihood')
plt.title('Data Points Colored by Log-likelihood', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X_with_outliers[~anomalies, 0], X_with_outliers[~anomalies, 1], 
           c='blue', alpha=0.6, s=50, label='Normal')
plt.scatter(X_with_outliers[anomalies, 0], X_with_outliers[anomalies, 1], 
           c='red', alpha=0.8, s=100, marker='x', label='Anomaly')
plt.title('Anomaly Detection using GMM', fontsize=14)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()

plt.tight_layout()
plt.show()

print(f"Number of detected anomalies: {np.sum(anomalies)}")

In [ ]:
# Generate new samples from GMM
gmm_gen = GaussianMixture(n_components=3, covariance_type='full', random_state=42)
gmm_gen.fit(X_elliptical)

# Generate new samples
X_generated, y_generated = gmm_gen.sample(200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.scatter(X_elliptical[:, 0], X_elliptical[:, 1], c='blue', alpha=0.5, s=30, label='Original')
ax1.set_title('Original Data', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.legend()

ax2 = axes[1]
ax2.scatter(X_elliptical[:, 0], X_elliptical[:, 1], c='blue', alpha=0.3, s=30, label='Original')
ax2.scatter(X_generated[:, 0], X_generated[:, 1], c='red', alpha=0.7, s=50, label='Generated')
ax2.set_title('Original + Generated Samples', fontsize=14)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.legend()

plt.suptitle('GMM as a Generative Model', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3D visualization of Iris GMM clustering
pca = PCA(n_components=3)
X_iris_3d = pca.fit_transform(X_iris_scaled)

fig = px.scatter_3d(x=X_iris_3d[:, 0], y=X_iris_3d[:, 1], z=X_iris_3d[:, 2],
                    color=labels_iris_gmm.astype(str),
                    title='GMM Clustering on Iris Dataset (3D PCA)',
                    labels={'x': 'PC1', 'y': 'PC2', 'z': 'PC3', 'color': 'Cluster'})
fig.update_layout(width=800, height=600)
fig.show()

<a id='conclusion'></a>
## 9. Conclusion

### Summary

In this notebook, we explored **Gaussian Mixture Models (GMM)** for clustering.

### Key Findings:

1. **Soft Clustering:** GMM provides probability of belonging to each cluster, unlike K-Means which gives hard assignments.

2. **Flexible Cluster Shapes:** GMM can model elliptical clusters through different covariance types.

3. **Model Selection:** BIC and AIC help select optimal number of components and covariance type.

4. **Covariance Types:**
   - **Full:** Most flexible, best for complex data
   - **Diagonal:** Axis-aligned ellipses
   - **Spherical:** Similar to K-Means
   - **Tied:** Shared covariance across components

5. **Applications:**
   - Clustering with uncertainty quantification
   - Density estimation
   - Anomaly detection
   - Generative modeling

### When to Use GMM over K-Means:
- Clusters have different shapes/orientations
- Need probability estimates
- Need to generate new samples
- Anomaly detection required

### References:
- Bishop, C. M. (2006). "Pattern Recognition and Machine Learning"
- Reynolds, D. (2009). "Gaussian Mixture Models"

In [ ]:
# Final summary
print("=" * 70)
print("        GAUSSIAN MIXTURE MODELS - FINAL SUMMARY")
print("=" * 70)
print("\n✓ Implemented GMM clustering using scikit-learn")
print("✓ Explored soft clustering with probability assignments")
print("✓ Compared 4 covariance types: full, tied, diagonal, spherical")
print("✓ Used BIC/AIC for model selection")
print("✓ Compared GMM vs K-Means on different datasets")
print("✓ Demonstrated anomaly detection using GMM")
print("✓ Used GMM as a generative model")
print("✓ Evaluated using multiple clustering quality metrics")
print("\n" + "=" * 70)